# 01. Data Loading and Merging

**Goal**: Load raw CSV files (Users, Telemetry, Death Events), remove PII, map death events to telemetry windows, and merge user metadata.

**Inputs**:
- `data/telemetry_phase_2.users.csv`
- `data/telemetry_phase_2.telemetries.csv`
- `data/telemetry_phase_2.deathevents.csv`

**Outputs**:
- `data/processed/merged_telemetry.csv` (Enriched with death counts and user data)
- `data/processed/merged_deathevents.csv` (Enriched with user data)

In [1]:
import pandas as pd
import numpy as np
import os

# File Paths
USERS_PATH = 'data/telemetry_phase_2.users.csv'
TELEMETRY_PATH = 'data/telemetry_phase_2.telemetries.csv'
DEATHEVENTS_PATH = 'data/telemetry_phase_2.deathevents.csv'

# Output Paths
PROCESSED_DIR = 'data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

OUTPUT_TELEMETRY = os.path.join(PROCESSED_DIR, 'merged_telemetry.csv')
OUTPUT_DEATHEVENTS = os.path.join(PROCESSED_DIR, 'merged_deathevents.csv')

## 1. Load Data

In [2]:
# Load Datasets
df_users = pd.read_csv(USERS_PATH)
df_telemetry = pd.read_csv(TELEMETRY_PATH)
df_deaths = pd.read_csv(DEATHEVENTS_PATH)

print(f"Loaded Users: {df_users.shape}")
print(f"Loaded Telemetry: {df_telemetry.shape}")
print(f"Loaded Death Events: {df_deaths.shape}")

Loaded Users: (61, 8)
Loaded Telemetry: (3492, 30)
Loaded Death Events: (250, 7)


## 2. Remove PII from Users

In [3]:
# Columns to drop
pii_cols = ['firstName', 'lastName', 'email', 'password', 'username'] 

# Drop only columns that exist
cols_to_drop = [c for c in pii_cols if c in df_users.columns]
df_users_clean = df_users.drop(columns=cols_to_drop)

print("Columns removed:", cols_to_drop)
display(df_users_clean.head())

Columns removed: ['firstName', 'lastName', 'email']


,_id,consentAgreed,apiKey,createdAt,__v
0,69747994eb8c4eacdf21a48d,True,key_08qs304ok,2026-01-24T13:19:40.456Z,0
1,6974892348d53c4152cf1421,True,key_2tknxe0by,2026-01-24T14:26:03.703Z,0
2,6974abcf315a39e91bc1e471,True,key_18cyf33bj,2026-01-24T16:53:59.160Z,0
3,6974acd8315a39e91bc1e52f,True,key_xzxtpmd15,2026-01-24T16:58:24.250Z,0
4,6974f3cc5d8e98b89f0a5d07,True,key_s0xtyyti1,2026-01-24T22:01:08.806Z,0


## 3. Map Death Events to Telemetry Windows

**Logic**:
1. Treat each telemetry row as a fixed 30-second behavioural window.
2. For each death event, find the telemetry window of the same user that occurs within the **next 30 seconds** after the death timestamp (`Death_Time <= Telemetry_Time < Death_Time + 30`).
3. Increment `death_count` on that telemetry row.
4. Do not create new rows.

In [4]:
# Filter users
target_user_ids = set(df_users_clean['_id'].unique())
df_telemetry = df_telemetry[df_telemetry['userId'].isin(target_user_ids)].copy()
df_deaths = df_deaths[df_deaths['userId'].isin(target_user_ids)].copy()

# Ensure timestamps are numeric
# FIX: Use 'timestamp' instead of 'time' based on CSV headers
df_telemetry['timestamp'] = pd.to_numeric(df_telemetry['timestamp'], errors='coerce')
df_deaths['timestamp'] = pd.to_numeric(df_deaths['timestamp'], errors='coerce')

# Initialize death_count
df_telemetry['death_count'] = 0

# Sort for efficient searching 
df_telemetry.sort_values(by=['userId', 'timestamp'], inplace=True)
df_deaths.sort_values(by=['userId', 'timestamp'], inplace=True)

print("Mapping deaths to telemetry windows...")
count_mapped = 0

# Group telemetry by user for faster lookup
telemetry_grouped = df_telemetry.groupby('userId')

for user_id, user_deaths in df_deaths.groupby('userId'):
    if user_id not in telemetry_grouped.groups:
        continue
    
    # Get user's telemetry indices in the main DF
    user_tele_indices = telemetry_grouped.groups[user_id]
    user_tele_times = df_telemetry.loc[user_tele_indices, 'timestamp'].values
    
    for death_time in user_deaths['timestamp']:
        if pd.isna(death_time):
             continue
             
        # Condition: Death_Time <= Tele_Time < Death_Time + 30 (Assuming timestamp is seconds?)
        # NOTE: 30-second window assumption. Check units. 
        # If timestamp is milliseconds (common in JS/Mongo), 30s = 30000ms.
        # Assuming seconds based on context. If rawJson has duration in seconds.
        # But if timestamp is standard epoch... usually seconds or ms.
        
        # Let's check magnitude during runtime or assume seconds based on earlier 30s mention.
        # The earlier code multiplied count * 30... so likely each row IS a 30s bucket.
        # So the timestamp is likely the START or END of that bucket?
        # The prompt said "find the telemetry window... within the next 30 seconds after the death timestamp"
        # Logic: death_time <= tele_time < death_time + 30
        
        mask = (user_tele_times >= death_time) & (user_tele_times < death_time + 30)
        
        matched_indices_local = np.where(mask)[0]
        
        if len(matched_indices_local) > 0:
            # Increment count on the first valid window found
            match_idx = user_tele_indices[matched_indices_local[0]]
            df_telemetry.at[match_idx, 'death_count'] += 1
            count_mapped += 1

print(f"Mapped {count_mapped} death events to telemetry windows.")

Mapping deaths to telemetry windows...
Mapped 0 death events to telemetry windows.


## 4. Merge User Data into Telemetry and Death Events

In [5]:
# Assuming Users table has '_id' which corresponds to 'userId' in others
user_key = '_id' if '_id' in df_users_clean.columns else 'userId'
telemetry_key = 'userId'
death_key = 'userId'

# Merge Telemetry
df_telemetry_merged = pd.merge(
    df_telemetry,
    df_users_clean,
    left_on=telemetry_key,
    right_on=user_key,
    how='left'
)

# Merge Death Events
df_deaths_merged = pd.merge(
    df_deaths,
    df_users_clean,
    left_on=death_key,
    right_on=user_key,
    how='left'
)

print(f"Merged Telemetry Shape: {df_telemetry_merged.shape}")
print(f"Merged Death Events Shape: {df_deaths_merged.shape}")

Merged Telemetry Shape: (3492, 36)
Merged Death Events Shape: (249, 12)


## 5. Save Processed Data

In [6]:
df_telemetry_merged.to_csv(OUTPUT_TELEMETRY, index=False)
df_deaths_merged.to_csv(OUTPUT_DEATHEVENTS, index=False)

print(f"Saved merged telemetry to: {OUTPUT_TELEMETRY}")
print(f"Saved merged deaths to: {OUTPUT_DEATHEVENTS}")

Saved merged telemetry to: data/processed\merged_telemetry.csv
Saved merged deaths to: data/processed\merged_deathevents.csv
